# 07 — Results, plots, errors and runtime analysis

**Purpose:** Create report-ready tables and charts using only completed and validated training and inference records.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


This notebook does not invent missing metrics or training times. Plots appear only when result files exist. **Do not mix validation and test metrics**, or compare different task label sets as if they were the same experiment.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import Counter
RUNS=read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl')
runs=pd.DataFrame([x for x in RUNS if not x.get('pilot') and x.get('status')=='complete'])
if len(runs):
    display(runs[['run_id','method','examples','lora_rank','seed','trainable_pct','wall_seconds','peak_gpu_allocated_gib']])
else: print('No completed training results yet.')

In [ ]:
paths=sorted((ROOT/'outputs/metrics').glob('*_leaderboard.csv'))
if paths:
    scores=pd.concat([pd.read_csv(x) for x in paths],ignore_index=True)
    display(scores[['run_id','lang','split','accuracy','macro_f1','invalid_count']])
    for key,group in scores.groupby(['task','lang','split']):
        ax=group.plot(kind='bar',x='run_id',y='macro_f1',legend=False,figsize=(max(7,len(group)*0.8),4))
        ax.set_title('Macro-F1 | '+str(key));ax.set_ylim(0,1);ax.set_ylabel('Macro-F1');
        plt.xticks(rotation=35,ha='right');plt.tight_layout()
        plt.savefig(ROOT/'outputs/plots'/('_'.join(key)+'_macro_f1.png'),dpi=160); plt.show()
else: print('No evaluated metrics yet. Run 05 then 06.')

## Paired prediction comparison

For a scientific comparison, models must be evaluated on the same test examples. This cell prints correct/incorrect disagreements for a pair. It does not claim significance from one seed or a tiny test set.

In [ ]:
COMPARE_A=None # example: 'sib200_lora_ur_n10_s42_r8_ur_test.csv'
COMPARE_B=None # example: 'sib200_full_ur_n10_s42_ur_test.csv'
if COMPARE_A and COMPARE_B:
    pa=ROOT/'outputs/predictions'/COMPARE_A; pb=ROOT/'outputs/predictions'/COMPARE_B
    ma,mb=read_json(pa.with_suffix('.meta.json')),read_json(pb.with_suffix('.meta.json'))
    if (ma['task'],ma['lang'],ma['split'],ma['dataset_sha256']) != (mb['task'],mb['lang'],mb['split'],mb['dataset_sha256']):
        raise ValueError('Different tasks/languages/splits/dataset hashes')
    da=pd.DataFrame(read_csv(pa)).set_index('example_id');db=pd.DataFrame(read_csv(pb)).set_index('example_id')
    assert set(da.index)==set(db.index),'Incomplete example alignment'
    aligned=da.join(db,lsuffix='_A',rsuffix='_B')
    a=(aligned['prediction_A']==aligned['reference_A'])
    b=(aligned['prediction_B']==aligned['reference_B'])
    print('Only A correct:',int((a & ~b).sum()), 'Only B correct:',int((~a & b).sum()))
    diff=aligned[a!=b]
    display(diff.head(30))
else: print('Set COMPARE_A/COMPARE_B to two complete prediction files.')

## Examine incorrectly classified Urdu examples

Review 20–50 errors manually for topic ambiguity, input length, misunderstood words, and invalid labels.

In [ ]:
ERROR_PREDICTION_FILE=None
if ERROR_PREDICTION_FILE:
    p=ROOT/'outputs/predictions'/ERROR_PREDICTION_FILE
    meta=read_json(p.with_suffix('.meta.json'))
    validate_prediction_metadata(meta,meta['task'],meta['lang'],meta['split'])
    reference=pd.DataFrame(load_task(meta['task'],meta['lang'],meta['split']))
    predicted=pd.DataFrame(read_csv(p))
    merged=predicted.merge(reference[['example_id','text','label']],on='example_id',validate='one_to_one')
    errors=merged[merged.prediction!=merged.label]
    errors[['example_id','text','label','prediction','raw_prediction']].head(50).to_csv(ROOT/'outputs/metrics'/'error_analysis_sample.csv',index=False)
    display(errors[['text','label','prediction','raw_prediction']].head(30))
else: print('Choose a validated inference file.')

## Actual GPU duration and memory (not predicted numbers)

In [ ]:
if len(runs):
    f=runs.copy(); f['minutes']=f.wall_seconds/60
    ax=f.plot(kind='bar',x='run_id',y='minutes',legend=False,figsize=(max(7,len(f)*0.8),4))
    ax.set_ylabel('Actual completed training minutes');plt.xticks(rotation=35,ha='right');plt.tight_layout()
    plt.savefig(ROOT/'outputs/plots'/'actual_train_times.png',dpi=160);plt.show()
else: print('Training time graphs require completed runs.')